In [1]:
!pip install torch

In [63]:
import torch
import torch.nn as nn
import torch.optim as optim 
import numpy as np 
import math

# T-net Model 

In [3]:
# inpput = sample (n x (d+1)) 
## here the d = dmax 
## for testing consoidering the d = 1 and n = 100
n = 100
d = 1
x = torch.zeros(n,d+1)
x.shape

torch.Size([100, 2])

In [38]:
class Tnet(nn.Module):
    def __init__(self,d_max,n,e):
        super().__init__()
        self.bn1 = nn.BatchNorm1d(e)
        self.bn2 = nn.BatchNorm1d(2*e)
        self.bn3 = nn.BatchNorm1d(4*e)
        self.shared = nn.Sequential(
            nn.Linear(d_max+1,e),
            nn.ReLU(),
            nn.Linear(e,2*e),
            nn.ReLU(),
            nn.Linear(2*e,4*e),
            nn.ReLU()
        )
        self.mlp = nn.Sequential(
            nn.Linear(4*e,2*e),
            nn.ReLU(),
            nn.Linear(2*e,1*e),
            nn.ReLU()
        )
    def forward(self,x):
        B,P,D = x.size()
        x = x.view(-1, D) # dim = (3200,2)
        #print(x.shape)
        x = self.shared[0](x) # Linear 1
        x = self.bn1(x)
        x = self.shared[1](x) # ReLU
        x = self.shared[2](x) # Linear 2
        x = self.bn2(x)
        x = self.shared[3](x) # ReLU
        x = self.shared[4](x) # Linear 3
        x = self.bn3(x)
        x = self.shared[5](x) # ReLU
        x = x.view(B,P,-1)
        #print(x.shape)
        pool,_ = torch.max(x,dim=1) # dim = 1 then the shape is (batch,..)
        #print(pool.shape)
        return self.mlp(pool)

In [39]:
e = 100
net = Tnet(d,n,e)
net

Tnet(
  (bn1): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm1d(400, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (shared): Sequential(
    (0): Linear(in_features=2, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=200, bias=True)
    (3): ReLU()
    (4): Linear(in_features=200, out_features=400, bias=True)
    (5): ReLU()
  )
  (mlp): Sequential(
    (0): Linear(in_features=400, out_features=200, bias=True)
    (1): ReLU()
    (2): Linear(in_features=200, out_features=100, bias=True)
    (3): ReLU()
  )
)

In [40]:
x = torch.zeros(32,n,d+1)
net(x).shape


torch.Size([32, 100])

In [41]:
x = net(x)

# GPT

In [22]:
x.unsqueeze(0).shape

torch.Size([1, 100])

## Token Embedding

In [54]:
# Tokens: [SOS, sin, (, x, +] => shape (T x e)
class TokenEmbedding(nn.Module):
    def __init__(self,voacb_size,e):
        super().__init__()
        self.embed = nn.Embedding(voacb_size,e)
    def forward(self,x):
        return self.embed(x)


In [61]:
vocab_size = 1000
number_tokens = 10
tokens = torch.ones(32,number_tokens).long()
token = TokenEmbedding(vocab_size,e)
token

TokenEmbedding(
  (embed): Embedding(1000, 100)
)

In [71]:
token(tokens).shape
x = token(tokens)

## Positional Embedding

In [76]:
class PositionalEmbedding(nn.Module):
    def __init__(self,e,tokens):
        super().__init__()
        self.d_k = e
        pe = torch.zeros(tokens,e)
        positions = torch.arange(0,tokens).float().unsqueeze(1)
        div = torch.exp(
            torch.arange(0,self.d_k,2).float() * -(math.log(1000.0)/self.d_k)
        )
        pe[:,0::2] = torch.sin(positions*div)
        pe[:,1::2] = torch.cos(positions*div)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe",pe)
    def forward(self,x):
        return x + self.pe[:,:x.shape[1],:].requires_grad_(False)

In [77]:
positonal = PositionalEmbedding(e,number_tokens)
positonal

PositionalEmbedding()

In [83]:
positonal(x).shape
x = positonal(x)

## Point Embedding 

In [86]:
x.expand(-1, number_tokens, -1) .shape

torch.Size([32, 10, 100])

## Transformer

In [103]:
class MaskMultiHeadAttention(nn.Module):
  def __init__(self,embedding_dim,heads):
    super().__init__()
    self.d_k = embedding_dim
    assert  embedding_dim % heads == 0
    self.heads = heads
    self.embed = embedding_dim // heads
    self.Q = nn.Linear(self.d_k,self.d_k)
    self.K = nn.Linear(self.d_k,self.d_k)
    self.V = nn.Linear(self.d_k,self.d_k)

  def forward(self,X,mask):
    batch_size,seq_len,embedding_dim = X.size()
    #print("Multihead masked attention")
    q = self.Q(X).view(batch_size,seq_len,self.heads,self.embed).transpose(2,1)
    k = self.K(X).view(batch_size,seq_len,self.heads,self.embed).transpose(2,1)
    v = self.V(X).view(batch_size,seq_len,self.heads,self.embed).transpose(2,1)
    k_t = k.transpose(-1,-2)
    s = (q @ k_t)
    if mask :
      mat = torch.zeros(s.size())
      mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
      mat[...,mask] = float("-inf")
      s = s + mat
    s = s / math.sqrt(self.embed) #the embedding dim in the multihead attention
    attention_scores = torch.nn.functional.softmax(s,dim=-1) @ v
    return attention_scores.transpose(2,1).contiguous().view(batch_size,seq_len,embedding_dim)

In [98]:
class LayerNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.ones(1))
        self.eps = 1e-2
    def forward(self,x):
        mean = torch.mean(x,dim=-1,keepdim=True)
        std = torch.std(x,dim=-1,keepdim=True)
        return self.alpha * ((x - mean)/(std + self.eps)) + self.beta

In [99]:
class FeedForward(nn.Module):
    def __init__(self,e):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(e,2048),
            nn.ReLU(),
            nn.Linear(2048,e),
            nn.ReLU()
        )
        
    def forward(self,x):
        return self.layer(x)

In [104]:
class transformerBlock(nn.Module):
    def __init__(self,e):
        super().__init__()
        self.attention = MaskMultiHeadAttention(e,10)
        self.norm = LayerNorm()
        self.fnn = FeedForward(e)
    def forward(self,x):
        x = self.norm(self.attention(x,1) + x)
        x = self.norm(self.fnn(x) + x)
        return x 

In [101]:
transformer = transformerBlock(e)
transformer

transformerBlock(
  (attention): MaskMultiHeadAttention(
    (Q): Linear(in_features=100, out_features=100, bias=True)
    (K): Linear(in_features=100, out_features=100, bias=True)
    (V): Linear(in_features=100, out_features=100, bias=True)
  )
  (norm): LayerNorm()
  (fnn): FeedForward(
    (layer): Sequential(
      (0): Linear(in_features=100, out_features=2048, bias=True)
      (1): ReLU()
      (2): Linear(in_features=2048, out_features=100, bias=True)
      (3): ReLU()
    )
  )
)

In [102]:
transformer(x).shape

Multihead masked attention


torch.Size([32, 10, 100])

In [105]:
class Encoder(nn.Module):
    def __init__(self,layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNorm()
    def forward(self,x):
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)

In [130]:
class transposed(nn.Module):
    def __init__(self,vocab_size,e):
        super().__init__()
        self.tran = nn.Linear(100, 1000, bias=False) 
    def forward(self,x):
        return self.tran(x)

In [131]:
class GPT(nn.Module):
    def __init__(self,e,d_max,transformer:Encoder,tokens_len,vocab_size):
        super().__init__()
        self.e = e
        self.d_max = d_max
        self.tokens_len = tokens_len
        self.transformer = transformer
        self.positional = PositionalEmbedding(e,tokens_len)
        self.token_emebed = TokenEmbedding(vocab_size,e)
        self.trans = transposed(vocab_size,e)
        
    def forward(self,x,tokens):
        tokens = self.token_emebed(tokens)
        tokens = self.positional(tokens)
        x = x.unsqueeze(1).expand(-1,self.tokens_len,-1)
        x = x + tokens
        x = self.transformer(x)
        print(x.shape)
        x = self.trans(x)
        return torch.softmax(x,dim=-1)


In [132]:
block = transformerBlock(e)
transformer = Encoder(nn.ModuleList(6*[block]))
gpt = GPT(e,d,transformer,number_tokens,vocab_size)
gpt

GPT(
  (transformer): Encoder(
    (layers): ModuleList(
      (0-5): 6 x transformerBlock(
        (attention): MaskMultiHeadAttention(
          (Q): Linear(in_features=100, out_features=100, bias=True)
          (K): Linear(in_features=100, out_features=100, bias=True)
          (V): Linear(in_features=100, out_features=100, bias=True)
        )
        (norm): LayerNorm()
        (fnn): FeedForward(
          (layer): Sequential(
            (0): Linear(in_features=100, out_features=2048, bias=True)
            (1): ReLU()
            (2): Linear(in_features=2048, out_features=100, bias=True)
            (3): ReLU()
          )
        )
      )
    )
    (norm): LayerNorm()
  )
  (positional): PositionalEmbedding()
  (token_emebed): TokenEmbedding(
    (embed): Embedding(1000, 100)
  )
  (trans): transposed(
    (tran): Linear(in_features=100, out_features=1000, bias=False)
  )
)

In [133]:
tokens = torch.ones(32,number_tokens).long()
x = torch.zeros(32,n,d+1)
x = net(x)
print(x.shape)
gpt(x,tokens).shape


torch.Size([32, 100])
torch.Size([32, 10, 100])


torch.Size([32, 10, 1000])

In [134]:
tokens.shape

torch.Size([32, 10])

In [135]:
out = gpt(x,tokens)

torch.Size([32, 10, 100])


In [136]:
out.shape

torch.Size([32, 10, 1000])

In [138]:
out[0].shape

torch.Size([10, 1000])

In [142]:
torch.sum(out[0][1])

tensor(1.0000, grad_fn=<SumBackward0>)

## Gpt complete

## SymbolicGpt

In [147]:
class SymbolicGPT(nn.Module):
    def __init__(self,net:Tnet,gpt:GPT):
        super().__init__()
        self.net = net
        self.gpt = gpt
    def forward(self,x,tokens):
        x = self.net(x)
        return self.gpt(x,tokens)

In [148]:
symba = SymbolicGPT(net,gpt)
symba

SymbolicGPT(
  (net): Tnet(
    (bn1): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (bn2): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (bn3): BatchNorm1d(400, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shared): Sequential(
      (0): Linear(in_features=2, out_features=100, bias=True)
      (1): ReLU()
      (2): Linear(in_features=100, out_features=200, bias=True)
      (3): ReLU()
      (4): Linear(in_features=200, out_features=400, bias=True)
      (5): ReLU()
    )
    (mlp): Sequential(
      (0): Linear(in_features=400, out_features=200, bias=True)
      (1): ReLU()
      (2): Linear(in_features=200, out_features=100, bias=True)
      (3): ReLU()
    )
  )
  (gpt): GPT(
    (transformer): Encoder(
      (layers): ModuleList(
        (0-5): 6 x transformerBlock(
          (attention): MaskMultiHeadAttention(
            (Q): Linear(in_features=100, out_features=100, bias=True)

In [149]:
x = torch.zeros(32,n,d+1)
tokens = torch.ones(32,number_tokens).long()
symba(x,tokens).shape

torch.Size([32, 10, 100])


torch.Size([32, 10, 1000])

## Code building done !!!!